<a href="https://colab.research.google.com/github/slowanimals/CSE-144-Final-Project/blob/main/v2_Kaush_Final_Project_CSE_144.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Starter Notebook (Student Version)

Fill in the sections marked with:

```python
# ========== YOUR CODE STARTS HERE ==========
# ========== YOUR CODE ENDS HERE ============
```

Do **not** change the rubric section above.

In [1]:
# If you are running in a fresh environment, uncomment to install dependencies:
# !pip -q install torch torchvision matplotlib tqdm scikit-learn

import os
import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split, Dataset
import kagglehub
from google.colab import userdata
import json
from torchvision.transforms import v2
from google.colab import drive
from copy import deepcopy

print("torch:", torch.__version__)
device = "cuda"  # For fixed, reproducible results. (You may switch to "cuda" after you finish debugging.)
print("device:", device)

def set_seed(seed: int = 42):
    """Make results as reproducible as possible across runs."""
    import os, random
    import numpy as np
    import torch

    os.environ["PYTHONHASHSEED"] = str(seed)
    # If you later switch to CUDA and want maximal determinism:
    # os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Deterministic flags (safe on CPU; on GPU some ops may error if non-deterministic)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    try:
        torch.use_deterministic_algorithms(True)
    except Exception as e:
        print("Warning: could not enable full deterministic algorithms:", e)

set_seed(42)


torch: 2.11.0+cu128
device: cuda


In [2]:
import os, gc
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print(torch.cuda.memory_allocated() / 1e9, "GB allocated")
print(torch.cuda.memory_reserved() / 1e9, "GB reserved")

0.0 GB allocated
0.0 GB reserved


In [3]:
print(os.path.exists("/content/drive/MyDrive/best.pt"))
os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump({"username": "kaushmendonca", "key": userdata.get("KAGGLE")}, f)
os.chmod("/root/.kaggle/kaggle.json", 0o600)

False


In [ ]:
key = userdata.get("KAGGLE")
print(repr(key))  # should print your key string, not None

'KGAT_b8adeaecf32736c5a59d6bb64c849338'


In [4]:
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


## 1. Dataset and DataLoader (15 pts)

In [5]:
# Q1: Build the data pipeline
batch_size = 16
num_workers = 0
os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump({"username": "kaushmendonca", "key": userdata.get("KAGGLE")}, f)
os.chmod("/root/.kaggle/kaggle.json", 0o600)

path = kagglehub.competition_download('ucsc-cse-144-spring-2026-final-project')

# ========== YOUR CODE STARTS HERE ==========
from PIL import Image

class TestDataset(Dataset):
    def __init__(self, folder, transform=None):
        self.folder = folder
        self.transform = transform
        self.images = sorted(os.listdir(folder))
    def __len__(self):
        return len(self.images)
    def __getitem__(self, idx):
        img = Image.open(os.path.join(self.folder, self.images[idx])).convert("RGB")
        return self.transform(img) if self.transform else img

train_tf = transforms.Compose([
    transforms.Resize((384,384)),
    transforms.RandomResizedCrop(380, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    # transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    # transforms.RandomPerspective(distortion_scale=0.2, p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.5),   # randomly blacks out patches

])
test_tf = transforms.Compose([
    transforms.Resize((380,380)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

full_train = datasets.ImageFolder(root=f"{path}/train", transform=train_tf)
val_dataset = deepcopy(full_train)
val_dataset.transform = test_tf
test_set   = TestDataset(folder=f"{path}/test", transform=test_tf)

train_size = int(0.8 * len(full_train))
val_size   = len(full_train) - train_size
indices = torch.randperm(len(full_train)).tolist()
train_indices = indices[:train_size]
val_indices   = indices[train_size:]

train_set = torch.utils.data.Subset(full_train, train_indices)
val_set   = torch.utils.data.Subset(val_dataset, val_indices)

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,  num_workers=num_workers)
val_loader   = DataLoader(val_set,   batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader  = DataLoader(test_set,  batch_size=batch_size, shuffle=False, num_workers=num_workers)
# ========== YOUR CODE ENDS HERE ============

print("train/val/test:", len(train_set), len(val_set), len(test_set))
print("Classes:", full_train.classes)

100%|██████████| 120M/120M [00:08<00:00, 15.0MB/s]

Extracting files...


train/val/test: 863 216 1036
Classes: ['0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '4', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '5', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '6', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '7', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '8', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '9', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99']


## 2. Define the CNN Model (20 pts)

In [10]:
from torch.nn.modules.pooling import MaxPool2d
from torch.nn.modules.batchnorm import BatchNorm2d

# from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights
from torchvision import models

class CNN(nn.Module):
    """Input: (B,1,28,28) -> Output: (B,10)"""
    def __init__(self, num_classes=100):
        super().__init__()
        # ========== YOUR CODE STARTS HERE ==========
        # Build a CNN with:
        # - Feature extraction: 3 convolutional blocks
        #   * First block: 1 input channel to 32 output channels, with BatchNorm and ReLU, then MaxPool
        #   * Second block: 32 to 64 channels, with BatchNorm and ReLU, then MaxPool
        #   * Third block: 64 to 128 channels, with ReLU (no pooling)
        #   Use kernel_size=3 and padding=1 for all convolutions, pool_size=2 for pooling
        # - Classifier: Flatten, then fully connected layers
        #   * After two MaxPool layers, your feature map will be 7x7
        #   * First FC layer: 128*7*7 inputs to 256 outputs, with ReLU and Dropout(0.3)
        #   * Final FC layer: 256 to num_classes outputs
        self.model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)

        # freeze weights
        for param in self.model.parameters():
          param.requires_grad = False

        self.model.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(in_features = 1792, out_features=num_classes)
        )


    def forward(self, x):
        # ========== YOUR CODE STARTS HERE ==========
        # Pass input through features, then classifier
        x = self.model(x)
        return x
        # ========== YOUR CODE ENDS HERE ============

model = CNN(num_classes=100).to(device)

Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 242MB/s]


## 3. Training Loop (40 pts)

### 3.1 Training Function (15 pts)

In [13]:
# Q3.1: Training function
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=1000)

MIXUP_ALPHA = 0.2

cutmix = v2.CutMix(num_classes=100)
mixup = v2.MixUp(num_classes=100)
cutmix_or_mixup = v2.RandomChoice([cutmix, mixup])

def mixup_data(x, y):
    """Returns mixed inputs and both label sets with blend ratio lambda."""
    lam = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA) if MIXUP_ALPHA > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    return mixed_x, y, y[idx], lam

def train_one_epoch(model, loader):
    """Train for one epoch and return (avg_loss, accuracy)."""
    model.train()

    running_loss = 0
    accuracy = 0
    total_samples = 0


    for i, batch in enumerate(loader):
      inputs, labels = batch
      inputs, labels = mixup_data(inputs, labels)

      inputs = inputs.to(device)
      labels = labels.to(device)

      optimizer.zero_grad()
      forward = model(inputs)
      loss = criterion(forward, labels)
      loss.backward()
      optimizer.step()

      running_loss += loss.item() # loss per batch?
      accuracy += (torch.argmax(forward, dim=1) == torch.argmax(labels, dim=1)).sum()
      total_samples += labels.size(0)


    return (running_loss/total_samples), (accuracy/total_samples)
    # ========== YOUR CODE ENDS HERE ============

### 3.2 Validation Function (8 pts)

In [14]:
# Q3.2: Validation function
@torch.no_grad()
def evaluate(model, loader):
    """Evaluate model and return (avg_loss, accuracy)."""
    # ========== YOUR CODE STARTS HERE ==========
    # TODO:
    # - Set model to evaluation mode
    # - Loop through batches without computing gradients
    # - Compute loss and accuracy
    # - Return average loss and accuracy
    model.eval()


    running_loss = 0
    accuracy = 0
    total_samples = 0
    for i, batch in enumerate(loader):
      inputs, labels = batch
      inputs = inputs.to(device)
      labels = labels.to(device)

      forward = model(inputs)
      loss = criterion(forward, labels)

      running_loss += loss.item()
      accuracy += (torch.argmax(forward, dim=1) == labels).sum()
      total_samples += labels.size(0)

    return (running_loss/total_samples), (accuracy/total_samples)
    # ========== YOUR CODE ENDS HERE ============

### 3.3 Training Loop (11 pts)

In [15]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [16]:
# Q3.3: Training loop with checkpointing
epochs = 1000
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0
best_epoch = -1
ckpt_path = "/content/drive/MyDrive/best.pt"
os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)

# ========== YOUR CODE STARTS HERE ==========
# TODO:
# - Loop through epochs
# - Each epoch: train, validate, log metrics
# - Track best validation accuracy and save checkpoint when improved
# - Print training and validation metrics each epoch
# - For checkpoint: save a dictionary containing keys `model_state_dict` and `epoch`

#early stopping
patience = 15
no_improve = 0

checkpoint = {}
for i in range(epochs):
  train_res = train_one_epoch(model, train_loader)
  eval_res = evaluate(model, val_loader)

  print(f'Epoch: {i}')
  print(f'Train Loss: {train_res[0]}, Train Accuracy: {train_res[1]}')
  print(f'Val Loss: {eval_res[0]}, Val Accuracy: {eval_res[1]}')

  history["train_loss"].append(train_res[0])
  history["train_acc"].append(train_res[1].item())
  history["val_loss"].append(eval_res[0])
  history["val_acc"].append(eval_res[1].item())


  if eval_res[1] > best_val_acc:
    best_val_acc = eval_res[1]
    best_epoch = i
    no_improve = 0 # reset no improvement for early stopping

    checkpoint['model_state_dict'] = model.state_dict()
    checkpoint['epoch'] = i
    torch.save(checkpoint, ckpt_path)
  else:
    no_improve += 1

  if no_improve >= patience:
    print(f"Early stopping at epoch {i}")
    break
# ========== YOUR CODE ENDS HERE ============

print("Best val acc:", best_val_acc, "at epoch", best_epoch)
print("Saved to:", ckpt_path)

ValueError: too many values to unpack (expected 2)

# Continue train

In [ ]:
checkpoint = torch.load("/content/drive/MyDrive/best.pt")
curr = checkpoint['epoch']

weights = checkpoint['model_state_dict']
model.load_state_dict(weights)

<All keys matched successfully>

In [ ]:
# Train CONT
epochs = 1000
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0
best_epoch = -1
ckpt_path = "/content/drive/MyDrive/best.pt"
os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)

#early stopping
patience = 15
no_improve = 0

checkpoint = {}
for i in range(curr, epochs):
  train_res = train_one_epoch(model, train_loader)
  eval_res = evaluate(model, val_loader)

  print(f'Epoch: {i}')
  print(f'Train Loss: {train_res[0]}, Train Accuracy: {train_res[1]}')
  print(f'Val Loss: {eval_res[0]}, Val Accuracy: {eval_res[1]}')

  history["train_loss"].append(train_res[0])
  history["train_acc"].append(train_res[1].item())
  history["val_loss"].append(eval_res[0])
  history["val_acc"].append(eval_res[1].item())


  if eval_res[1] > best_val_acc:
    best_val_acc = eval_res[1]
    best_epoch = i
    no_improve = 0 # reset no improvement for early stopping

    checkpoint['model_state_dict'] = model.state_dict()
    checkpoint['epoch'] = i
    torch.save(checkpoint, ckpt_path)
  else:
    no_improve += 1

  if no_improve >= patience:
    print(f"Early stopping at epoch {i}")
    break
# ========== YOUR CODE ENDS HERE ============

print("Best val acc:", best_val_acc, "at epoch", best_epoch)
print("Saved to:", ckpt_path)

KeyboardInterrupt: 

6/1, 10:10am. Epoch 110,
Train Loss: 0.01397723905733535, Train Accuracy: 0.7114716172218323
Val Loss: 0.019031723340352375, Val Accuracy: 0.5370370149612427

6/1 2:10pm
Epoch: 71
Train Loss: 0.13467178042213998, Train Accuracy: 0.6326767206192017
Val Loss: 0.11943331526385413, Val Accuracy: 0.569444477558136

# fine tuning

In [ ]:
checkpoint = torch.load("/content/drive/MyDrive/best.pt")
model.load_state_dict(checkpoint['model_state_dict'])

for param in model.model.features[-3].parameters():
  param.requires_grad = True
for param in model.model.classifier.parameters():
  param.requires_grad = True

optimizer = torch.optim.AdamW([
    # discriminative decay
      {"params": model.model.features[-3].parameters(), "lr":1e-3},
      {"params": model.model.features[-2].parameters(), "lr":1e-3},
      {"params": model.model.classifier.parameters(), "lr":1e-2},
    ],
    weight_decay=0.01
)
# scheduler will adjust learning rate over time
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

In [ ]:
new_ckpt_path = "/content/drive/MyDrive/ft_best.pt"
os.makedirs(os.path.dirname(new_ckpt_path), exist_ok=True)

epochs = 1000
patience = 20
no_improve = 0
best_val_acc = 0.0

checkpoint = {}
for i in range(epochs):
  train_res = train_one_epoch(model, train_loader)
  eval_res = evaluate(model, val_loader)
  scheduler.step() # adjust LR

  print(f'Epoch: {i}')
  print(f'Train Loss: {train_res[0]}, Train Accuracy: {train_res[1]}')
  print(f'Val Loss: {eval_res[0]}, Val Accuracy: {eval_res[1]}')

  history["train_loss"].append(train_res[0])
  history["train_acc"].append(train_res[1].item())
  history["val_loss"].append(eval_res[0])
  history["val_acc"].append(eval_res[1].item())


  if eval_res[1] > best_val_acc:
    best_val_acc = eval_res[1]
    best_epoch = i
    no_improve = 0 # reset no improvement for early stopping

    checkpoint['model_state_dict'] = model.state_dict()
    checkpoint['epoch'] = i
    torch.save(checkpoint, new_ckpt_path)
    print("Improved!")
  else:
    no_improve += 1

  if no_improve >= patience:
    print(f"Early stopping at epoch {i}")
    break

print("Best val acc:", best_val_acc, "at epoch", best_epoch)
print("Saved to:", new_ckpt_path)

Epoch: 0
Train Loss: 0.14984352301335693, Train Accuracy: 0.5353417992591858
Val Loss: 0.10959602174935518, Val Accuracy: 0.5324074029922485
Improved!
Epoch: 1
Train Loss: 0.14880615465886898, Train Accuracy: 0.5654692649841309
Val Loss: 0.10518410112018939, Val Accuracy: 0.5370370149612427
Improved!
Epoch: 2
Train Loss: 0.12813354125580947, Train Accuracy: 0.623406708240509
Val Loss: 0.10341324353659595, Val Accuracy: 0.5879629850387573
Improved!
Epoch: 3
Train Loss: 0.12137735418345119, Train Accuracy: 0.658169150352478
Val Loss: 0.10017213694475315, Val Accuracy: 0.5925925970077515
Improved!
Epoch: 4
Train Loss: 0.11191290575549058, Train Accuracy: 0.6882966160774231
Val Loss: 0.08790896584590276, Val Accuracy: 0.6157407164573669
Improved!
Epoch: 5
Train Loss: 0.1186709834277837, Train Accuracy: 0.6871379017829895
Val Loss: 0.09358401624140916, Val Accuracy: 0.6157407164573669
Epoch: 6
Train Loss: 0.11323093532272728, Train Accuracy: 0.725376546382904
Val Loss: 0.09483686365463116, 

In [ ]:
new_curr = torch.load('/content/drive/MyDrive/ft_best.pt')


### Plot Training Curves

In [ ]:
# Q3.4: Plot curves
# ========== YOUR CODE STARTS HERE ==========
# TODO:
# - Create two plots: one for loss (train vs val), one for accuracy (train vs val)
# - Use the history dictionary to get the values
# - Add labels, legends, and display the plots
plt.plot(torch.arange(epochs), history["train_loss"], label="Train Loss")
plt.plot(torch.arange(epochs), history["val_loss"], label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title("Train Loss vs Val Loss")
plt.legend()
plt.show()

plt.plot(torch.arange(epochs), history["train_acc"], label="Train Accuracy")
plt.plot(torch.arange(epochs), history["val_acc"], label="Val Accuracy")
plt.title("Train Accuracy vs Val Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

# ========== YOUR CODE ENDS HERE ============

AttributeError: 'list' object has no attribute 'item'

## 4. Testing (5 pts)

In [ ]:
# Q4: Test evaluation
# ========== YOUR CODE STARTS HERE ==========
# TODO:
# - Load the best checkpoint (it's a dictionary with 'model_state_dict' and 'epoch')
# - Load the model state from the checkpoint
# - Evaluate on the test set
# - Print test loss and accuracy
model.load_state_dict(torch.load(new_ckpt_path)['model_state_dict']) # load parameters
final_eval = evaluate(model, test_loader)

test_loss, test_acc = final_eval[0], final_eval[1]
# ========== YOUR CODE ENDS HERE ============

print(f"Test loss: {test_loss:.4f} | Test acc: {test_acc:.4f}")

ValueError: too many values to unpack (expected 2)